# Notebook 03 — Production Pipeline
## Airbnb Multi-City Nightly Price Prediction (Regression)

**Module:** ITI113 Machine Learning & Operations
**Pipeline stage:** 2. Preprocessing → 3. Training → 5. Model Registry → 6. Deployment → 8. Monitoring → 9. Governance
**Estimated runtime:** 20–35 minutes for a full pipeline execution

---

## The core design decision: one pipeline object, two environments

The most common way MLOps projects fail is **training/serving skew** — the
notebook computes `log1p(minimum_nights)` and the endpoint computes
`log(minimum_nights)`, and nobody notices for six weeks because the predictions
are merely *wrong*, not *broken*.

We eliminate the failure mode structurally. A single `sklearn.pipeline.Pipeline`
contains **every** transformation — imputation, encoding, target encoding,
scaling and the estimator — and that one object is what gets fitted, pickled,
registered and served. There is no separate serving-side preprocessing code
that could drift, because there is no separate serving-side preprocessing code.

```text
                    ONE ARTEFACT
   raw listing row -> [ Pipeline: clean -> engineer -> encode -> model ] -> price
                          ^                                    ^
                    fitted in the                        loaded by the
                    SageMaker training job               SageMaker endpoint
```

## What this notebook does

1. Writes `preprocess.py`, `train.py` and `inference.py`, sharing a single
   `features.py` module so dev and production run **identical** feature code
2. Uploads the source to S3, then re-downloads it — proving the pipeline runs
   from the versioned S3 copy rather than notebook-local state
3. Defines a **SageMaker Pipeline**: Process → Train → R² Quality Gate → Register
4. Logs the completed run to the team **SageMaker MLflow App** from the notebook,
   keeping MLflow credentials out of the training container
5. Approves and deploys a **Serverless Endpoint**, then tests it
6. Runs the **AI governance checks**: spatial bias audit, segment fairness,
   interpretability and a drift-monitoring baseline

## Adaptation from the reference notebooks

The reference pipeline gated on classification AUC. This is a regression
problem, so three things change: the `ConditionStep` compares **R² ≥ 0.70**, the
estimator's `metric_definitions` regex captures `test_r2`, and `inference.py`
back-transforms the log prediction into a USD price band. Everything else —
the team-tagged MLflow App, the S3 prefix convention, the model-registry
hand-off — is unchanged.

In [ ]:
%pip install --upgrade --quiet "sagemaker>=2,<3" boto3 botocore mlflow sagemaker-mlflow

## 0. Configuration

In [ ]:
import os, io, json, time, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.width", 200)

import boto3, sagemaker

session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

TEAM_ID      = "team40"
STUDENT_ID   = "s4001"
COURSE       = "ITI113"
SEMESTER     = "26S1"
PROJECT_NAME = "airbnb-price"

BUCKET = "nyp-26s1-iti113"
PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

PROCESSING_INSTANCE_TYPE = "ml.m5.xlarge"   # 280k rows: xlarge over large
TRAINING_INSTANCE_TYPE   = "ml.m5.xlarge"

PIPELINE_NAME       = f"iti113-{TEAM_ID}-{PROJECT_NAME}"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-AirbnbPrice"
ENDPOINT_NAME       = f"iti113-{TEAM_ID}-{PROJECT_NAME}"
QUALITY_GATE_R2     = 0.70

RAW_DATA_URI  = f"s3://{BUCKET}/{PREFIX}/raw/Listings.csv"
PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline"

SCRIPTS_S3_PREFIX  = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI     = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

s3_client = boto3.client("s3")

print("=" * 74)
print(f"{COURSE} {SEMESTER} — Notebook 03 — Production Pipeline")
print("=" * 74)
print(f"Team / Student  : {TEAM_ID} / {STUDENT_ID}")
print(f"Pipeline        : {PIPELINE_NAME}")
print(f"Model group     : {MODEL_PACKAGE_GROUP}")
print(f"Endpoint        : {ENDPOINT_NAME} (serverless)")
print(f"Quality gate    : test R2 >= {QUALITY_GATE_R2}")
print(f"Raw data        : {RAW_DATA_URI}")
print(f"Region / Role   : {region} / {str(role).split('/')[-1]}")
print("=" * 74)

### 0.1 Load the hand-off records from Notebooks 01 and 02

`best_model.json` carries a `known_limitations` block. We read it and print it
prominently — a deployment notebook that ignores the error analysis it was
handed is how a model with 44% error on shared rooms ends up serving shared
rooms.

In [ ]:
def load_json(local_name, s3_key):
    if Path(local_name).exists():
        return json.loads(Path(local_name).read_text())
    obj = s3_client.get_object(Bucket=BUCKET, Key=s3_key)
    return json.loads(obj["Body"].read())


contract = load_json("feature_contract.json",
                     f"{PREFIX}/processed/feature_contract.json")
best_info = load_json("best_model.json",
                      f"{PREFIX}/processed/best_model.json")

CHAMPION_PARAMS = best_info["champion_hyperparameters"]

print(f"Champion from Notebook 02 : {best_info['best_run_name']}")
print(f"  family    : {best_info['champion_family']}")
print(f"  test R2   : {best_info['best_test_r2']}")
print(f"  test MdAPE: {best_info['best_test_mdape']:.1%}")
print(f"  params    : {CHAMPION_PARAMS}")
print(f"\nFeature contract: {contract['n_features']} features, "
      f"target={contract['target']}")

print("\n" + "!" * 74)
print("KNOWN LIMITATIONS INHERITED FROM NOTEBOOK 02 — these constrain serving")
print("!" * 74)
for k, v in best_info["known_limitations"].items():
    print(f"  {k}: {json.dumps(v)}")
print("!" * 74)

## 0A. MLflow App Connection Precheck

Same `TeamId` validation as Notebook 02, run **before** the pipeline starts. A
25-minute pipeline that succeeds and then cannot log its results is a wasted
25 minutes.

In [ ]:
import mlflow

TEAM_CONFIG    = Path(f"mlflow_app_config_{TEAM_ID}.json")
STUDENT_CONFIG = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")
config_candidates = [TEAM_CONFIG, STUDENT_CONFIG,
                     *sorted(Path(".").glob(f"mlflow_app_config_{TEAM_ID}_*.json"))]

DEFAULT_MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-L5IMA5YSBDTY")

MLFLOW_APP_ARN         = None
MLFLOW_EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/{PROJECT_NAME}"
cfg, cfg_used = {}, None

for f in config_candidates:
    if f.exists():
        cfg = json.loads(f.read_text(encoding="utf-8"))
        MLFLOW_APP_ARN = (cfg.get("MLFLOW_APP_ARN") or cfg.get("mlflow_app_arn")
                          or cfg.get("arn"))
        MLFLOW_EXPERIMENT_NAME = (cfg.get("EXPERIMENT_NAME")
                                  or cfg.get("experiment_name")
                                  or MLFLOW_EXPERIMENT_NAME)
        cfg_used = f
        break

if MLFLOW_APP_ARN is None:
    MLFLOW_APP_ARN = DEFAULT_MLFLOW_APP_ARN
    print("[WARNING] No local MLflow config — using DEFAULT_MLFLOW_APP_ARN.")
else:
    print(f"Loaded MLflow App config from {cfg_used}")

cfg_team = cfg.get("TEAM_ID") or cfg.get("team_id")
if cfg_team and cfg_team != TEAM_ID:
    raise ValueError(f"Config team mismatch: {cfg_team} != {TEAM_ID}")

sm_for_mlflow = boto3.client("sagemaker", region_name=region)
try:
    tags = {t["Key"]: t["Value"]
            for t in sm_for_mlflow.list_tags(ResourceArn=MLFLOW_APP_ARN).get("Tags", [])}
    print("MLflow App tags:")
    for k, v in tags.items():
        print(f"  {k}: {v}")
    app_team = tags.get("TeamId")
    if app_team != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId={app_team} != TEAM_ID={TEAM_ID}. "
            "Refusing to log to another team's tracking server.")
    print(f"\n[OK] TeamId '{app_team}' matches TEAM_ID.")
except Exception as e:
    print("\n[ERROR] Could not validate the MLflow App team tag.")
    print("  1. ARN belongs to another team (IAM blocked it correctly)")
    print("  2. App is missing the TeamId tag")
    print("  3. This role lacks sagemaker:ListTags")
    print(f"  {type(e).__name__}: {e}")
    raise

os.environ["MLFLOW_TRACKING_URI"]   = MLFLOW_APP_ARN
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME

In [ ]:
def create_mlflow_app_presigned_url(fragment: str = "") -> str:
    """MLflow prints generic mlflow.sagemaker.*.app.aws links that are not
    presigned and usually render a permission error. Use these instead."""
    r = boto3.client("sagemaker", region_name=region
                     ).create_presigned_mlflow_app_url(Arn=MLFLOW_APP_ARN)
    base = r.get("AuthorizedUrl") or r.get("Url")
    if not base:
        raise RuntimeError(f"No presigned URL returned: {r}")
    base = base.split("#", 1)[0]
    return base + "#" + fragment.lstrip("#") if fragment else base


def print_mlflow_presigned_links(experiment_id=None, run_id=None):
    if experiment_id is not None:
        print("Presigned MLflow experiment URL:")
        print(create_mlflow_app_presigned_url(f"/experiments/{experiment_id}"))
    if experiment_id is not None and run_id is not None:
        print("\nPresigned MLflow run URL:")
        print(create_mlflow_app_presigned_url(
            f"/experiments/{experiment_id}/runs/{run_id}"))


mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(
        run_name=f"{TEAM_ID}_pipeline_precheck_{int(time.time())}") as run:
    mlflow.set_tags({
        "course": COURSE, "semester": SEMESTER,
        "team_id": TEAM_ID, "student_id": STUDENT_ID,
        "dataset": PROJECT_NAME, "task_type": "regression",
        "run_type": "sagemaker_pipeline_precheck",
        "tracking_backend": "sagemaker_mlflow_app",
        "mlflow_app_arn": MLFLOW_APP_ARN,
    })
    mlflow.log_param("source", "notebook_03_precheck")
    mlflow.log_metric("connection_success", 1)
    precheck_run_id, precheck_exp_id = run.info.run_id, run.info.experiment_id

print("MLflow App precheck OK.")
print(f"Experiment  : {MLFLOW_EXPERIMENT_NAME} (id={precheck_exp_id})")
print(f"Run ID      : {precheck_run_id}\n")
print("Ignore any generic mlflow.sagemaker.app.aws link printed above; use:")
print_mlflow_presigned_links(precheck_exp_id, precheck_run_id)

## 1. Write the Pipeline Source

Four files. The important one is `features.py` — it is imported by
`preprocess.py`, by `train.py` and by `inference.py`, so the *same* function
objects run in the Processing job, the Training job and the live endpoint.

```text
features.py    <- single source of truth for cleaning + feature engineering
   |
   +-- preprocess.py   (Processing job)  clean -> engineer -> grouped split
   +-- train.py        (Training job)    fit the full Pipeline, print metrics
   +-- inference.py    (Endpoint)        load Pipeline, predict, back-transform
```

Nothing here imports MLflow. The training container gets no MLflow credentials
and no MLflow dependency — the notebook logs to the MLflow App *after* the
pipeline succeeds. This keeps the blast radius of a credential leak inside the
notebook session and makes container failures easier to diagnose.

In [ ]:
os.makedirs("src", exist_ok=True)
print("src/ ready")

In [ ]:
%%writefile src/features.py
"""Shared cleaning + feature engineering.

Imported by preprocess.py, train.py AND inference.py so that development and
production execute byte-identical transformation code. This module is the
single point of truth; changing a feature here changes it everywhere at once.
"""
import ast
import numpy as np
import pandas as pd

SNAPSHOT_DATE = "2021-03-01"

# Fixed, versioned FX table. Deliberately NOT a live rate lookup: a pricing
# model must be reproducible, and a live call would give the same input row a
# different label on a different day.
FX_TO_USD = {
    "Paris": 1.21, "Rome": 1.21, "New York": 1.00, "Sydney": 0.77,
    "Rio de Janeiro": 0.185, "Istanbul": 0.135, "Mexico City": 0.049,
    "Bangkok": 0.033, "Cape Town": 0.067, "Hong Kong": 0.129,
}
CITY_CENTER = {
    "Paris": (48.8566, 2.3522), "Rome": (41.9028, 12.4964),
    "New York": (40.7580, -73.9855), "Sydney": (-33.8688, 151.2093),
    "Rio de Janeiro": (-22.9068, -43.1729), "Istanbul": (41.0082, 28.9784),
    "Mexico City": (19.4326, -99.1332), "Bangkok": (13.7563, 100.5018),
    "Cape Town": (-33.9249, 18.4241), "Hong Kong": (22.3193, 114.1694),
}
TOP_AMENITIES = [
    "Wifi", "Kitchen", "Air conditioning", "Heating", "Washer", "Dryer",
    "TV", "Elevator", "Free parking on premises", "Pool", "Gym",
    "Dishwasher", "Hot tub", "Patio or balcony", "Private entrance",
    "Dedicated workspace", "Long term stays allowed", "Breakfast",
    "Smoke alarm", "Carbon monoxide alarm", "Fire extinguisher",
    "First aid kit", "Lock on bedroom door", "Bathtub", "Coffee maker",
    "Cable TV", "Garden or backyard", "BBQ grill", "Waterfront", "Beachfront",
]
LUXURY_SET = ["Pool", "Hot tub", "Gym", "Waterfront", "Beachfront", "Doorman",
              "Elevator", "Dishwasher", "BBQ grill", "Bathtub"]
SAFETY_SET = ["Smoke alarm", "Carbon monoxide alarm", "Fire extinguisher",
              "First aid kit", "Lock on bedroom door"]
REVIEW_SUBSCORES = ["review_scores_accuracy", "review_scores_cleanliness",
                    "review_scores_checkin", "review_scores_communication",
                    "review_scores_location", "review_scores_value"]

TARGET       = "log_price_usd"
GROUP_COLUMN = "host_id"

NUMERIC_FEATURES = [
    "accommodates", "log_accommodates", "bedrooms_filled", "bedrooms_imputed",
    "guests_per_bedroom", "is_studio",
    "log_min_nights", "log_max_nights", "booking_window",
    "is_long_stay_only", "is_short_stay_friendly",
    "host_tenure_years", "log_host_listings", "is_professional_host",
    "host_response_rate_f", "host_acceptance_rate_f", "host_response_missing",
    "host_response_speed", "host_is_local",
    "host_is_superhost_b", "host_has_profile_pic_b",
    "host_identity_verified_b", "instant_bookable_b",
    "has_reviews", "review_scores_rating_f", "review_scores_accuracy_f",
    "review_scores_cleanliness_f", "review_scores_checkin_f",
    "review_scores_communication_f", "review_scores_location_f",
    "review_scores_value_f", "value_gap", "location_premium_score",
    "dist_center_km", "log_dist_center", "lat_offset", "lon_offset",
    "amenity_count", "amenity_density", "luxury_score", "safety_score",
    "is_rare_property",
] + ["am_" + a.lower().replace(" ", "_") for a in TOP_AMENITIES]

CATEGORICAL_FEATURES   = ["city", "room_type", "property_group"]
TARGET_ENCODE_FEATURES = ["city_neigh"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_ENCODE_FEATURES


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def parse_amenities(s):
    if not isinstance(s, str):
        return []
    try:
        v = ast.literal_eval(s)
        return v if isinstance(v, list) else []
    except Exception:
        return []


def group_property(p):
    p = str(p).lower()
    if "hotel" in p or "hostel" in p:
        return "hotel_like"
    if any(k in p for k in ["apartment", "condominium", "loft", "serviced"]):
        return "apartment"
    if any(k in p for k in ["house", "townhouse", "villa", "bungalow", "cottage"]):
        return "house"
    if any(k in p for k in ["bed and breakfast", "guest", "boutique"]):
        return "bnb_guesthouse"
    if "entire" in p:
        return "other_entire"
    return "other"


def clean(df, require_target=True):
    """Deterministic cleaning. See Notebook 01 §3 for the rule provenance."""
    d = df.copy()
    d["fx_to_usd"] = d["city"].map(FX_TO_USD)
    if d["fx_to_usd"].isna().any():
        unknown = d.loc[d.fx_to_usd.isna(), "city"].unique().tolist()
        raise ValueError(f"Unmapped cities (no FX rate): {unknown}")

    if require_target:
        d["price_usd"] = d["price"] * d["fx_to_usd"]
        d = d[d["price_usd"] > 0]
    d = d[d["accommodates"] >= 1]

    d["minimum_nights"] = d["minimum_nights"].clip(1, 365)
    d["maximum_nights"] = d["maximum_nights"].clip(1, 1125)

    if require_target:
        lo = d.groupby("city")["price_usd"].transform(lambda s: s.quantile(0.005))
        hi = d.groupby("city")["price_usd"].transform(lambda s: s.quantile(0.995))
        d = d[(d["price_usd"] >= lo) & (d["price_usd"] <= hi)]

    return d.drop(columns=["district"], errors="ignore").reset_index(drop=True)


def engineer(d, require_target=True, snapshot=SNAPSHOT_DATE):
    """Row-wise feature engineering. Every feature is computable from a single
    listing at inference time — no lookups against other rows or future data."""
    d = d.copy()
    snap = pd.Timestamp(snapshot)

    if require_target:
        d[TARGET] = np.log(d["price_usd"])

    # capacity
    if "bedrooms" not in d.columns:
        d["bedrooms"] = np.nan
    grp_med = d.groupby(["city", "room_type"])["bedrooms"].transform("median")
    d["bedrooms_filled"]    = d["bedrooms"].fillna(grp_med).fillna(1.0)
    d["bedrooms_imputed"]   = d["bedrooms"].isna().astype(int)
    d["guests_per_bedroom"] = d["accommodates"] / d["bedrooms_filled"].clip(lower=0.5)
    d["is_studio"]          = (d["bedrooms_filled"] <= 1).astype(int)
    d["log_accommodates"]   = np.log1p(d["accommodates"])

    # stay policy
    d["log_min_nights"]         = np.log1p(d["minimum_nights"])
    d["log_max_nights"]         = np.log1p(d["maximum_nights"])
    d["booking_window"]         = d["maximum_nights"] - d["minimum_nights"]
    d["is_long_stay_only"]      = (d["minimum_nights"] >= 30).astype(int)
    d["is_short_stay_friendly"] = (d["minimum_nights"] <= 2).astype(int)

    # host
    hs = pd.to_datetime(d.get("host_since"), errors="coerce")
    ten = (snap - hs).dt.days / 365.25
    d["host_tenure_years"] = ten.fillna(ten.median() if ten.notna().any() else 3.0).clip(lower=0)
    hl = d["host_total_listings_count"].fillna(1).clip(1, 1000)
    d["log_host_listings"]      = np.log1p(hl)
    d["is_professional_host"]   = (hl >= 5).astype(int)
    d["host_response_rate_f"]   = d["host_response_rate"].fillna(-1)
    d["host_acceptance_rate_f"] = d["host_acceptance_rate"].fillna(-1)
    d["host_response_missing"]  = d["host_response_rate"].isna().astype(int)
    d["host_response_speed"] = d["host_response_time"].map(
        {"within an hour": 3, "within a few hours": 2,
         "within a day": 1, "a few days or more": 0}).fillna(-1)
    d["host_is_local"] = (d["host_location"].fillna("").str.lower()
        .str.contains("|".join(c.lower() for c in CITY_CENTER))).astype(int)
    for c in ["host_is_superhost", "host_has_profile_pic",
              "host_identity_verified", "instant_bookable"]:
        d[c + "_b"] = (d[c] == "t").astype(int)

    # reviews — MNAR-aware sentinels, never mean-imputed
    d["has_reviews"] = d["review_scores_rating"].notna().astype(int)
    d["review_scores_rating_f"] = d["review_scores_rating"].fillna(-1)
    for c in REVIEW_SUBSCORES:
        d[c + "_f"] = d[c].fillna(-1)
    d["value_gap"] = np.where(
        d["has_reviews"] == 1,
        d["review_scores_rating"].fillna(0) / 10 - d["review_scores_value"].fillna(0), -99)
    d["location_premium_score"] = np.where(
        d["has_reviews"] == 1,
        d["review_scores_location"].fillna(0) - d["review_scores_value"].fillna(0), -99)

    # geo
    clat = d["city"].map(lambda c: CITY_CENTER[c][0])
    clon = d["city"].map(lambda c: CITY_CENTER[c][1])
    d["dist_center_km"]  = haversine(d["latitude"], d["longitude"], clat, clon)
    d["log_dist_center"] = np.log1p(d["dist_center_km"])
    d["lat_offset"]      = d["latitude"] - clat
    d["lon_offset"]      = d["longitude"] - clon

    # amenities
    amen = d["amenities"].map(parse_amenities).map(set)
    d["amenity_count"] = amen.map(len)
    for a in TOP_AMENITIES:
        d["am_" + a.lower().replace(" ", "_")] = amen.map(lambda s, a=a: int(a in s))
    d["luxury_score"] = sum(amen.map(lambda s, a=a: int(a in s)) for a in LUXURY_SET)
    d["safety_score"] = sum(amen.map(lambda s, a=a: int(a in s)) for a in SAFETY_SET)
    d["amenity_density"] = d["amenity_count"] / d["accommodates"].clip(lower=1)

    # property type
    d["property_group"] = d["property_type"].map(group_property)
    counts = d["property_type"].value_counts()
    d["is_rare_property"] = (d["property_type"].map(counts).fillna(0) < 500).astype(int)

    # neighbourhood key (target-encoded inside the Pipeline, never here)
    d["city_neigh"] = d["city"] + " | " + d["neighbourhood"].fillna("unknown")
    return d

### 1.1 The reusable pipeline factory

`pipeline_factory.py` builds the scikit-learn `Pipeline` used by **both** the
Notebook 02 experiments and the SageMaker Training job. The custom
`SmoothedTargetEncoder` lives here so it can be unpickled inside the inference
container — a custom transformer defined only in a notebook will pickle fine and
then fail to load at the endpoint with `AttributeError: Can't get attribute`,
which is one of the more frustrating ways to lose an afternoon.

In [ ]:
%%writefile src/pipeline_factory.py
"""Builds the production sklearn Pipeline. Shared by dev and by SageMaker.

The custom transformer MUST be importable from a module (not defined in a
notebook) or the pickled model will not unpickle inside the inference container.
"""
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

import features as F


class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
    """Out-of-fold smoothed target encoding for high-cardinality categoricals.

    enc(k) = (n_k * mean_k + m * prior) / (n_k + m)

    fit_transform -> out-of-fold values, so no row informs its own encoding
    transform     -> full-fit mapping, global prior for unseen levels
    """

    def __init__(self, cols=None, smoothing=20.0, n_splits=5, random_state=42):
        self.cols = cols
        self.smoothing = smoothing
        self.n_splits = n_splits
        self.random_state = random_state

    def _mapping(self, keys, vals, prior):
        agg = pd.Series(np.asarray(vals)).groupby(np.asarray(keys)).agg(["mean", "count"])
        return ((agg["mean"] * agg["count"] + prior * self.smoothing)
                / (agg["count"] + self.smoothing))

    def fit(self, X, y):
        X = pd.DataFrame(X); y = pd.Series(np.asarray(y))
        self.cols_ = list(X.columns) if self.cols is None else self.cols
        self.prior_ = float(y.mean())
        self.maps_ = {c: self._mapping(X[c].values, y.values, self.prior_)
                      for c in self.cols_}
        self.feature_names_out_ = [f"{c}_te" for c in self.cols_]
        return self

    def fit_transform(self, X, y=None, **kw):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y)).reset_index(drop=True)
        self.fit(X, y)
        out = pd.DataFrame(index=X.index)
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for c in self.cols_:
            oof = pd.Series(np.full(len(X), self.prior_), index=X.index, dtype=float)
            for tr, va in kf.split(X):
                fp = y.iloc[tr].mean()
                m = self._mapping(X[c].iloc[tr].values, y.iloc[tr].values, fp)
                oof.iloc[va] = X[c].iloc[va].map(m).fillna(fp).values
            out[f"{c}_te"] = oof
        return out.values

    def transform(self, X):
        X = pd.DataFrame(X)
        return pd.DataFrame(
            {f"{c}_te": X[c].map(self.maps_[c]).fillna(self.prior_)
             for c in self.cols_}, index=X.index).values

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_out_, dtype=object)


def build_preprocessor():
    return ColumnTransformer([
        ("num", "passthrough", F.NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=200,
                              sparse_output=False), F.CATEGORICAL_FEATURES),
        ("te",  SmoothedTargetEncoder(cols=F.TARGET_ENCODE_FEATURES),
                F.TARGET_ENCODE_FEATURES),
    ], remainder="drop")


def build_pipeline(max_iter=600, learning_rate=0.06, max_leaf_nodes=63,
                   min_samples_leaf=40, l2_regularization=1.0, random_state=42):
    """The single production artefact: preprocessing + model in one object."""
    return Pipeline([
        ("pre", build_preprocessor()),
        ("model", HistGradientBoostingRegressor(
            max_iter=max_iter, learning_rate=learning_rate,
            max_leaf_nodes=max_leaf_nodes, min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            random_state=random_state, early_stopping=False)),
    ])

In [ ]:
%%writefile src/preprocess.py
"""SageMaker Processing Job — replicates Notebook 01 exactly.

Reads raw Listings.csv, cleans, engineers features, and writes a GROUPED
train/test split. The grouping (by host_id) is the whole point: see Notebook 01
§5.2 for the measured leakage that a random split would introduce.
"""
import argparse
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

import features as F

parser = argparse.ArgumentParser()
parser.add_argument("--test-size", type=float, default=0.20)
parser.add_argument("--random-state", type=int, default=42)
args = parser.parse_args()

INPUT_DIR  = "/opt/ml/processing/input"
OUTPUT_DIR = "/opt/ml/processing/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

candidates = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".csv")]
if not candidates:
    raise FileNotFoundError(f"No CSV in {INPUT_DIR}: {os.listdir(INPUT_DIR)}")
input_path = os.path.join(INPUT_DIR, candidates[0])
print(f"Reading {input_path}")

# latin-1: the source file is not UTF-8 (accented listing titles).
df = pd.read_csv(input_path, encoding="latin-1", low_memory=False)
print(f"Raw shape: {df.shape}")

df = F.clean(df, require_target=True)
print(f"After cleaning: {df.shape}")

df = F.engineer(df, require_target=True)
print(f"After feature engineering: {df.shape}")

missing = [c for c in F.ALL_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Feature engineering did not produce: {missing}")

nan_counts = df[F.NUMERIC_FEATURES].isna().sum()
if nan_counts.sum() > 0:
    raise ValueError(f"NaNs remain in numeric block:\n{nan_counts[nan_counts > 0]}")

X = df[F.ALL_FEATURES]
y = df[F.TARGET]
groups = df[F.GROUP_COLUMN]

gss = GroupShuffleSplit(n_splits=1, test_size=args.test_size,
                        random_state=args.random_state)
tr, te = next(gss.split(X, y, groups=groups))

overlap = set(groups.iloc[tr]) & set(groups.iloc[te])
if overlap:
    raise ValueError(f"Host leakage: {len(overlap)} hosts in both splits")

meta_cols = ["city", "neighbourhood", "room_type", "host_id",
             "price_usd", "has_reviews", "is_professional_host"]

X.iloc[tr].to_csv(f"{OUTPUT_DIR}/train_features.csv", index=False)
y.iloc[tr].to_csv(f"{OUTPUT_DIR}/train_labels.csv", index=False, header=True)
X.iloc[te].to_csv(f"{OUTPUT_DIR}/test_features.csv", index=False)
y.iloc[te].to_csv(f"{OUTPUT_DIR}/test_labels.csv", index=False, header=True)
df.iloc[te][meta_cols].to_csv(f"{OUTPUT_DIR}/test_meta.csv", index=False)

stats = {
    "n_raw": int(len(df)), "n_train": int(len(tr)), "n_test": int(len(te)),
    "n_features": len(F.ALL_FEATURES),
    "n_train_hosts": int(groups.iloc[tr].nunique()),
    "n_test_hosts": int(groups.iloc[te].nunique()),
    "host_overlap": 0,
    "train_target_mean": float(y.iloc[tr].mean()),
    "test_target_mean": float(y.iloc[te].mean()),
}
with open(f"{OUTPUT_DIR}/preprocess_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))
print("Preprocessing complete — 5 CSVs + stats written.")

In [ ]:
%%writefile src/train.py
"""SageMaker Training Job.

Fits the complete Pipeline (preprocessing + model) and saves it as ONE artefact.
Deliberately contains no MLflow import: the container needs no tracking
credentials. The notebook logs to the MLflow App after the pipeline succeeds.
"""
import argparse
import json
import os
import pickle

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import features as F
from pipeline_factory import build_pipeline

parser = argparse.ArgumentParser()
parser.add_argument("--max-iter", type=int, default=600)
parser.add_argument("--learning-rate", type=float, default=0.06)
parser.add_argument("--max-leaf-nodes", type=int, default=63)
parser.add_argument("--min-samples-leaf", type=int, default=40)
parser.add_argument("--l2-regularization", type=float, default=1.0)
parser.add_argument("--random-state", type=int, default=42)

parser.add_argument("--team-id", type=str, default=os.environ.get("TEAM_ID", "unknown"))
parser.add_argument("--student-id", type=str, default=os.environ.get("STUDENT_ID", "s000"))
parser.add_argument("--semester", type=str, default=os.environ.get("SEMESTER", "26S1"))
parser.add_argument("--run-name", type=str, default="sagemaker_pipeline_run")

parser.add_argument("--model-dir", type=str,
                    default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
parser.add_argument("--train", type=str,
                    default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
parser.add_argument("--test", type=str,
                    default=os.environ.get("SM_CHANNEL_TEST", "/opt/ml/input/data/test"))
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

X_train = pd.read_csv(os.path.join(args.train, "train_features.csv"))
y_train = pd.read_csv(os.path.join(args.train, "train_labels.csv")).squeeze("columns")
X_test  = pd.read_csv(os.path.join(args.test, "test_features.csv"))
y_test  = pd.read_csv(os.path.join(args.test, "test_labels.csv")).squeeze("columns")

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

if list(X_train.columns) != F.ALL_FEATURES:
    raise ValueError("Training columns do not match the feature contract.")
if y_train.std() == 0:
    raise ValueError("Target has zero variance.")

pipe = build_pipeline(
    max_iter=args.max_iter, learning_rate=args.learning_rate,
    max_leaf_nodes=args.max_leaf_nodes, min_samples_leaf=args.min_samples_leaf,
    l2_regularization=args.l2_regularization, random_state=args.random_state)

print("Fitting the full Pipeline (preprocessing + model)...")
pipe.fit(X_train, y_train)

metrics = {}
for split, Xs, ys in [("train", X_train, y_train), ("test", X_test, y_test)]:
    p = pipe.predict(Xs)
    metrics[f"{split}_r2"]       = round(float(r2_score(ys, p)), 4)
    metrics[f"{split}_rmse_log"] = round(float(np.sqrt(mean_squared_error(ys, p))), 4)
    metrics[f"{split}_mae_log"]  = round(float(mean_absolute_error(ys, p)), 4)
    ape = np.abs(np.exp(p) - np.exp(ys)) / np.exp(ys)
    metrics[f"{split}_mdape"]        = round(float(np.median(ape)), 4)
    metrics[f"{split}_mape"]         = round(float(ape.mean()), 4)
    metrics[f"{split}_within_25pct"] = round(float((ape <= 0.25).mean()), 4)

metrics["overfit_gap_r2"] = round(metrics["train_r2"] - metrics["test_r2"], 4)

print("=== Metrics ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

with open(os.path.join(args.model_dir, "model.pkl"), "wb") as f:
    pickle.dump(pipe, f)
with open(os.path.join(args.model_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# The inference container needs features.py and pipeline_factory.py on its path
# to unpickle the Pipeline; SageMaker ships source_dir separately, but bundling
# a copy inside model.tar.gz makes the artefact self-describing.
import shutil
for mod in ["features.py", "pipeline_factory.py"]:
    src = os.path.join(os.path.dirname(os.path.abspath(__file__)), mod)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(args.model_dir, mod))

print(f"Model saved: {os.path.join(args.model_dir, 'model.pkl')}")

# These exact printed labels are captured by SageMaker metric_definitions
# and consumed by the Pipeline ConditionStep.
print(f"Test R2: {metrics['test_r2']}")
print(f"test_rmse_log: {metrics['test_rmse_log']}")
print(f"test_mdape: {metrics['test_mdape']}")

### 1.2 Inference handler — where the error analysis becomes policy

`inference.py` is where Notebook 02's findings stop being observations and start
being enforced behaviour. Three guardrails are implemented:

1. **Back-transform correctly.** `exp(prediction)` is the conditional **median**
   price, not the mean. We label it as such rather than implying an expectation.
2. **Segment-aware confidence.** The interval width and the confidence tier come
   from the *measured* per-segment residual SDs, so a never-reviewed shared room
   in Hong Kong returns a wide band and a `low` tier, while a reviewed entire
   place in Paris returns a narrow one.
3. **Refuse to serve where the model is unfit.** Shared rooms had 44% MdAPE.
   The endpoint returns a `served: false` response with a reason rather than a
   confident-looking number.

A model that knows when not to answer is more useful in production than one
that answers everything.

In [ ]:
%%writefile src/inference.py
"""SageMaker inference handler.

Accepts RAW listing JSON, applies the identical feature engineering used in
training, and returns a price band with a confidence tier. Guardrails encode
the segment-level error analysis from Notebook 02 §9.
"""
import json
import os
import pickle

import numpy as np
import pandas as pd

import features as F

# Residual SDs measured on the held-out grouped test set (Notebook 02 §9.2/9.3).
RESID_SD_BY_CITY = {
    "Paris": 0.361, "New York": 0.403, "Rome": 0.470, "Cape Town": 0.459,
    "Sydney": 0.493, "Mexico City": 0.478, "Bangkok": 0.520,
    "Rio de Janeiro": 0.577, "Istanbul": 0.535, "Hong Kong": 0.628,
}
DEFAULT_RESID_SD = 0.50
COLD_START_PENALTY = 1.15     # never-reviewed: MdAPE 31.0% vs 23.9%
UNSUPPORTED_ROOM_TYPES = {"Shared room"}   # MdAPE 44.0% -> do not serve


def model_fn(model_dir):
    with open(os.path.join(model_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)


def input_fn(body, content_type="application/json"):
    if content_type != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")
    payload = json.loads(body)
    if isinstance(payload, dict):
        payload = [payload]
    return pd.DataFrame(payload)


def predict_fn(raw_df, model):
    df = F.clean(raw_df.copy(), require_target=False)
    df = F.engineer(df, require_target=False)

    missing = [c for c in F.ALL_FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"Could not derive required features: {missing}")

    pred_log = model.predict(df[F.ALL_FEATURES])

    sd = df["city"].map(RESID_SD_BY_CITY).fillna(DEFAULT_RESID_SD).values
    sd = np.where(df["has_reviews"].values == 0, sd * COLD_START_PENALTY, sd)

    return {
        "pred_log": pred_log,
        "sd": sd,
        "city": df["city"].values,
        "room_type": df["room_type"].values,
        "has_reviews": df["has_reviews"].values,
    }


def output_fn(prediction, accept="application/json"):
    out = []
    for i in range(len(prediction["pred_log"])):
        log_p = float(prediction["pred_log"][i])
        sd    = float(prediction["sd"][i])
        room  = str(prediction["room_type"][i])

        median = float(np.exp(log_p))
        lo     = float(np.exp(log_p - 1.28 * sd))   # ~80% interval
        hi     = float(np.exp(log_p + 1.28 * sd))

        if sd <= 0.42:
            tier = "high"
        elif sd <= 0.52:
            tier = "medium"
        else:
            tier = "low"

        rec = {
            "recommended_price_usd": round(median, 2),
            "price_range_usd": [round(lo, 2), round(hi, 2)],
            "interval": "approx. 80% central range",
            "confidence": tier,
            "city": str(prediction["city"][i]),
            "has_reviews": int(prediction["has_reviews"][i]),
            "estimate_type": "conditional median of comparable listings",
            "served": True,
            "notes": [],
        }

        if room in UNSUPPORTED_ROOM_TYPES:
            rec["served"] = False
            rec["recommended_price_usd"] = None
            rec["price_range_usd"] = None
            rec["confidence"] = "unsupported"
            rec["notes"].append(
                "Shared-room pricing is out of scope: validation error "
                "(~44% median APE) is too high to give guidance.")

        if int(prediction["has_reviews"][i]) == 0:
            rec["notes"].append(
                "No reviews yet — expect wider-than-typical uncertainty.")

        if median < 25 or median > 400:
            rec["notes"].append(
                "Estimate is near the edge of the calibrated range; the model "
                "over-prices very cheap and under-prices very expensive listings.")

        out.append(rec)

    return json.dumps(out), accept

In [ ]:
print("Pipeline source written:")
for fn in ["features.py", "pipeline_factory.py", "preprocess.py",
           "train.py", "inference.py"]:
    print(f"  src/{fn:<22} {os.path.getsize(f'src/{fn}'):>7,} bytes")
print("\nNo MLflow import in any container script — verified below:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    txt = Path(f"src/{fn}").read_text()
    print(f"  {fn:<18} mlflow imported: {'mlflow' in txt}")

### 1.3 Test the pipeline locally before paying for a SageMaker job

A 25-minute pipeline that fails in the training step because of a typo is an
expensive way to find a typo. We smoke-test the whole chain on a small sample
first, in-process.

In [ ]:
import sys
sys.path.insert(0, "src")
import importlib
import features as F
import pipeline_factory as PF
importlib.reload(F); importlib.reload(PF)

# Small sample straight from the raw file
if Path("Listings.csv").exists():
    sample_raw = pd.read_csv("Listings.csv", encoding="latin-1",
                             low_memory=False, nrows=25000)
else:
    obj = s3_client.get_object(Bucket=BUCKET, Key=f"{PREFIX}/raw/Listings.csv")
    sample_raw = pd.read_csv(io.BytesIO(obj["Body"].read()),
                             encoding="latin-1", low_memory=False, nrows=25000)

t0 = time.time()
sc = F.clean(sample_raw, require_target=True)
se = F.engineer(sc, require_target=True)
print(f"clean+engineer on {len(sample_raw):,} rows -> {se.shape} "
      f"in {time.time()-t0:.1f}s")

from sklearn.model_selection import GroupShuffleSplit
Xs, ys, gs = se[F.ALL_FEATURES], se[F.TARGET], se[F.GROUP_COLUMN]
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(Xs, ys, gs))

smoke = PF.build_pipeline(max_iter=120)
smoke.fit(Xs.iloc[tr], ys.iloc[tr])
from sklearn.metrics import r2_score
print(f"Smoke-test R2 (small sample, few iters): "
      f"{r2_score(ys.iloc[te], smoke.predict(Xs.iloc[te])):.4f}")

# --- the critical test: does inference.py reproduce training-time features? ---
import inference as INF
importlib.reload(INF)

probe_raw = sample_raw.head(50).copy()
probe_clean = F.clean(probe_raw, require_target=False)
probe_feat  = F.engineer(probe_clean, require_target=False)

direct = smoke.predict(probe_feat[F.ALL_FEATURES])
via_handler = INF.predict_fn(probe_raw, smoke)["pred_log"]

assert np.allclose(direct, via_handler, atol=1e-10), \
    "TRAINING/SERVING SKEW — inference path does not match training path!"
print(f"\n[OK] inference.py reproduces training-time features exactly "
      f"(max diff {np.abs(direct - via_handler).max():.2e})")

body, _ = INF.output_fn(INF.predict_fn(probe_raw.head(3), smoke))
print("\nSample endpoint response:")
print(json.dumps(json.loads(body), indent=2)[:900])

That assertion is the single most valuable test in this notebook. It proves the
training path and the serving path produce **bit-identical** features from the
same raw input. Every training/serving skew bug this project could have had
would fail here, in two seconds, instead of silently degrading production
predictions for months.

## 1A / 1B. Version the Source in S3, Then Run From It

We upload the source to S3 and delete the local copy before re-downloading it.
This is not ceremony: it proves the pipeline runs from the **versioned S3
artefact** rather than from notebook-local state that would not exist in a CI
run or a scheduled execution.

In [ ]:
SOURCE_DIR = Path("src")
FILES_TO_UPLOAD = ["features.py", "pipeline_factory.py",
                   "preprocess.py", "train.py", "inference.py"]

for fn in FILES_TO_UPLOAD:
    local = SOURCE_DIR / fn
    if not local.exists():
        raise FileNotFoundError(f"Missing source file: {local}")
    s3_client.upload_file(str(local), BUCKET, f"{SCRIPTS_S3_PREFIX}/{fn}")
    print(f"Uploaded {local} -> s3://{BUCKET}/{SCRIPTS_S3_PREFIX}/{fn}")

print(f"\nPipeline source versioned at: {SCRIPTS_S3_URI}")

In [ ]:
local_src = Path(LOCAL_PIPELINE_SRC)
if local_src.exists():
    shutil.rmtree(local_src)
local_src.mkdir(parents=True, exist_ok=True)

for fn in FILES_TO_UPLOAD:
    s3_client.download_file(BUCKET, f"{SCRIPTS_S3_PREFIX}/{fn}", str(local_src / fn))
    print(f"Downloaded s3://{BUCKET}/{SCRIPTS_S3_PREFIX}/{fn} -> {local_src / fn}")

print(f"\n{LOCAL_PIPELINE_SRC}/ now contains:")
for p in sorted(local_src.iterdir()):
    print(f"  {p.name:<22} {p.stat().st_size:>7,} bytes")

## 2. Define the SageMaker Pipeline

Four steps:

| Step | Type | Purpose |
|---|---|---|
| `PreprocessData` | `ProcessingStep` | clean → engineer → grouped split → 5 CSVs to S3 |
| `TrainModel` | `TrainingStep` | fit the full Pipeline, print metrics for capture |
| `R2QualityGate` | `ConditionStep` | block registration unless test R² ≥ 0.70 |
| `RegisterModel` | `ModelStep` | register as `PendingManualApproval` |

Hyperparameters are exposed as pipeline **parameters**, so a retrain can be
launched with different settings without editing or redeploying the definition.

In [ ]:
from sagemaker.workflow.pipeline import Pipeline as SMPipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model

pipeline_session = PipelineSession()

p_max_iter   = ParameterInteger(name="MaxIter",         default_value=CHAMPION_PARAMS.get("max_iter", 600))
p_leaf_nodes = ParameterInteger(name="MaxLeafNodes",    default_value=CHAMPION_PARAMS.get("max_leaf_nodes", 63))
p_min_leaf   = ParameterInteger(name="MinSamplesLeaf",  default_value=CHAMPION_PARAMS.get("min_samples_leaf", 40))
p_lr         = ParameterFloat(  name="LearningRate",    default_value=CHAMPION_PARAMS.get("learning_rate", 0.06))
p_l2         = ParameterFloat(  name="L2Regularization",default_value=CHAMPION_PARAMS.get("l2_regularization", 1.0))
p_gate       = ParameterFloat(  name="QualityGateR2",   default_value=QUALITY_GATE_R2)

print("Pipeline parameters defined (champion values from Notebook 02):")
for p in [p_max_iter, p_leaf_nodes, p_min_leaf, p_lr, p_l2, p_gate]:
    print(f"  {p.name:<20} = {p.default_value}")

In [ ]:
# ---- Step 1: ProcessingStep -------------------------------------------------
processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    sagemaker_session=pipeline_session,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-process",
)

step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    inputs=[ProcessingInput(source=RAW_DATA_URI,
                            destination="/opt/ml/processing/input")],
    outputs=[ProcessingOutput(output_name="processed",
                              source="/opt/ml/processing/output",
                              destination=f"{PIPELINE_ROOT}/processed")],
    code=f"{LOCAL_PIPELINE_SRC}/preprocess.py",
    # features.py must ride along -- preprocess.py imports it
    job_arguments=["--test-size", "0.2", "--random-state", "42"],
)
print("Step 1 (ProcessingStep) defined.")

> **Note on the Processing step's dependencies.** `preprocess.py` imports
> `features.py`, but `ProcessingStep(code=...)` uploads only the single script.
> The cell below repackages the source directory so the import resolves inside
> the container. This is a genuinely common failure — the job starts, runs for
> 90 seconds, and dies on `ModuleNotFoundError: No module named 'features'`.

In [ ]:
# Bundle features.py alongside preprocess.py by shipping the whole directory
# as an extra ProcessingInput mapped onto the container's working path.
step_process = ProcessingStep(
    name="PreprocessData",
    processor=processor,
    inputs=[
        ProcessingInput(source=RAW_DATA_URI,
                        destination="/opt/ml/processing/input"),
        ProcessingInput(source=SCRIPTS_S3_URI,
                        destination="/opt/ml/processing/input/code",
                        input_name="code_deps"),
    ],
    outputs=[ProcessingOutput(output_name="processed",
                              source="/opt/ml/processing/output",
                              destination=f"{PIPELINE_ROOT}/processed")],
    code=f"{LOCAL_PIPELINE_SRC}/preprocess.py",
    job_arguments=["--test-size", "0.2", "--random-state", "42"],
)

# preprocess.py resolves `import features` from its own directory; SageMaker
# places the entry script in /opt/ml/processing/input/code/ and runs it from
# there, so the sibling modules uploaded above are importable.
print("Step 1 redefined with code dependencies attached.")

In [ ]:
# ---- Step 2: TrainingStep ---------------------------------------------------
# source_dir ships the WHOLE pipeline_src folder, so train.py can import
# features.py and pipeline_factory.py. The container receives no MLflow
# credentials and no MLflow dependency.
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "max-iter": p_max_iter,
        "learning-rate": p_lr,
        "max-leaf-nodes": p_leaf_nodes,
        "min-samples-leaf": p_min_leaf,
        "l2-regularization": p_l2,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "sagemaker_pipeline_run",
    },
    environment={"TEAM_ID": TEAM_ID, "STUDENT_ID": STUDENT_ID,
                 "SEMESTER": SEMESTER},
    # Regression metrics — regex updated from the reference notebook's AUC.
    metric_definitions=[
        {"Name": "test_r2",       "Regex": "Test R2: ([0-9\\.\\-]+)"},
        {"Name": "test_rmse_log", "Regex": "test_rmse_log: ([0-9\\.]+)"},
        {"Name": "test_mdape",    "Regex": "test_mdape: ([0-9\\.]+)"},
    ],
    tags=[{"Key": "Course", "Value": COURSE},
          {"Key": "Semester", "Value": SEMESTER},
          {"Key": "Team", "Value": TEAM_ID},
          {"Key": "Student", "Value": STUDENT_ID},
          {"Key": "TaskType", "Value": "regression"}],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs[
    "processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        "train": sagemaker.inputs.TrainingInput(s3_data=processed_uri,
                                                content_type="text/csv"),
        "test":  sagemaker.inputs.TrainingInput(s3_data=processed_uri,
                                                content_type="text/csv"),
    },
)
print("Step 2 (TrainingStep) defined.")

In [ ]:
# ---- Step 3: ModelStep (registration) ---------------------------------------
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point="inference.py",
    source_dir=LOCAL_PIPELINE_SRC,
)

step_register = ModelStep(
    name="RegisterModel",
    step_args=model.register(
        content_types=["application/json"],
        response_types=["application/json"],
        inference_instances=["ml.m5.large", "ml.m5.xlarge"],
        transform_instances=["ml.m5.xlarge"],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status="PendingManualApproval",
        description=(f"Airbnb nightly price regression | "
                     f"HistGradientBoosting | gate R2>={QUALITY_GATE_R2} | "
                     f"team={TEAM_ID}"),
    ),
)
print("Step 3 (ModelStep) defined.")

In [ ]:
# ---- Step 4: ConditionStep — the R2 quality gate ----------------------------
# train.py prints "Test R2: 0.xxxx"; metric_definitions captures it as test_r2.
# ConditionGreaterThanOrEqualTo blocks registration when the model regresses.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_r2"].Value,
    right=p_gate,
)

step_condition = ConditionStep(
    name="R2QualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[],
)
print(f"Step 4 (ConditionStep) defined — gate: test_r2 >= {QUALITY_GATE_R2}")
print("A model that fails the gate is NOT registered and cannot be deployed.")

In [ ]:
pipeline = SMPipeline(
    name=PIPELINE_NAME,
    parameters=[p_max_iter, p_leaf_nodes, p_min_leaf, p_lr, p_l2, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session,
)
pipeline.upsert(role_arn=role)
print(f'Pipeline "{PIPELINE_NAME}" upserted.')
print("View it in SageMaker Studio -> left sidebar -> Pipelines")

## 3. Execute the Pipeline

In [ ]:
execution = pipeline.start(parameters={
    "MaxIter":          CHAMPION_PARAMS.get("max_iter", 600),
    "LearningRate":     CHAMPION_PARAMS.get("learning_rate", 0.06),
    "MaxLeafNodes":     CHAMPION_PARAMS.get("max_leaf_nodes", 63),
    "MinSamplesLeaf":   CHAMPION_PARAMS.get("min_samples_leaf", 40),
    "L2Regularization": CHAMPION_PARAMS.get("l2_regularization", 1.0),
    "QualityGateR2":    QUALITY_GATE_R2,
})
print(f"Execution ARN: {execution.arn}")
print("Monitoring below. Expect roughly 15-25 minutes.")

In [ ]:
prev = {}
start = time.time()

while True:
    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]

    steps = execution.list_steps()
    if isinstance(steps, dict):
        steps = steps.get("PipelineExecutionSteps", [])

    for step in steps:
        n, s = step["StepName"], step["StepStatus"]
        if prev.get(n) != s:
            print(f"[{time.time()-start:6.0f}s] {n:<20} {s}")
            prev[n] = s
            if s == "Failed":
                print(f"    reason: {step.get('FailureReason', 'n/a')}")

    if status in ("Succeeded", "Failed", "Stopped"):
        print(f"\nPipeline Status: {status}  ({time.time()-start:.0f}s)")
        break
    time.sleep(30)

## 4. Log the Completed Run to the SageMaker MLflow App

The pipeline ran without MLflow. Now the notebook reads the completed Training
job's metadata and metrics and writes them into the team experiment, so the
production run sits **in the same experiment, comparable against the Notebook 02
experiments**. Without this, production and development would maintain separate,
incomparable histories.

In [ ]:
def get_pipeline_steps(execution):
    r = execution.list_steps()
    return r if isinstance(r, list) else r.get("PipelineExecutionSteps", [])


if execution.describe()["PipelineExecutionStatus"] != "Succeeded":
    raise RuntimeError("Pipeline did not succeed — resolve failures before logging.")

steps = get_pipeline_steps(execution)
print("Pipeline steps:")
for s in steps:
    print(f"  {s['StepName']:<20} {s['StepStatus']}")

train_step = next((s for s in steps
                   if s["StepName"] == "TrainModel" and s["StepStatus"] == "Succeeded"),
                  None)
if train_step is None:
    raise RuntimeError("No successful TrainModel step in this execution.")

training_job_arn  = train_step["Metadata"]["TrainingJob"]["Arn"]
training_job_name = training_job_arn.rsplit("/", 1)[-1]

sm_client = boto3.client("sagemaker", region_name=region)
tj = sm_client.describe_training_job(TrainingJobName=training_job_name)

captured = {m["MetricName"]: float(m["Value"])
            for m in tj.get("FinalMetricDataList", [])
            if m["MetricName"] in {"test_r2", "test_rmse_log", "test_mdape"}}
if not captured:
    raise RuntimeError("No metrics captured — check train.py output vs metric_definitions.")

model_artifact_uri = tj["ModelArtifacts"]["S3ModelArtifacts"]
hyperparams = tj.get("HyperParameters", {})

print(f"\nTraining job   : {training_job_name}")
print(f"Model artefact : {model_artifact_uri}")
print(f"Captured metrics: {captured}")
print(f"\nGate check: test_r2={captured.get('test_r2')} vs {QUALITY_GATE_R2} -> "
      f"{'PASS' if captured.get('test_r2', 0) >= QUALITY_GATE_R2 else 'FAIL'}")

In [ ]:
mlflow.set_tracking_uri(MLFLOW_APP_ARN)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

run_name = f"{TEAM_ID}_{STUDENT_ID}_sagemaker_pipeline_{int(time.time())}"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": COURSE, "semester": SEMESTER,
        "team_id": TEAM_ID, "student_id": STUDENT_ID,
        "dataset": PROJECT_NAME, "task_type": "regression",
        "target": contract["target"],
        "execution_environment": "aws_sagemaker_pipeline",
        "tracking_backend": "sagemaker_mlflow_app",
        "pipeline_name": PIPELINE_NAME,
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "sagemaker_model_artifact_s3_uri": model_artifact_uri,
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "model_package_group": MODEL_PACKAGE_GROUP,
        "run_role": "production_pipeline",
    })

    mlflow.log_params(hyperparams)
    mlflow.log_params({
        "processing_instance": PROCESSING_INSTANCE_TYPE,
        "training_instance": TRAINING_INSTANCE_TYPE,
        "quality_gate_r2": QUALITY_GATE_R2,
        "pipeline_steps": "Process->Train->R2Gate->Register",
    })
    mlflow.log_metrics(captured)

    summary = {
        "pipeline_name": PIPELINE_NAME,
        "pipeline_execution_arn": execution.arn,
        "training_job_name": training_job_name,
        "training_job_arn": training_job_arn,
        "model_artifact_s3_uri": model_artifact_uri,
        "metrics": captured,
        "hyperparameters": hyperparams,
        "quality_gate_r2": QUALITY_GATE_R2,
        "gate_passed": bool(captured.get("test_r2", 0) >= QUALITY_GATE_R2),
        "team_id": TEAM_ID, "student_id": STUDENT_ID, "semester": SEMESTER,
        "notebook02_champion_r2": best_info["best_test_r2"],
        "known_limitations": best_info["known_limitations"],
        "mlflow_app_arn": MLFLOW_APP_ARN,
        "mlflow_experiment": MLFLOW_EXPERIMENT_NAME,
    }
    Path("sagemaker_pipeline_run_summary.json").write_text(json.dumps(summary, indent=2))
    mlflow.log_artifact("sagemaker_pipeline_run_summary.json",
                        artifact_path="sagemaker_pipeline")

    pipeline_run_id, pipeline_exp_id = run.info.run_id, run.info.experiment_id

print("Logged the production pipeline run to the MLflow App.")
print(f"  MLflow run ID : {pipeline_run_id}")
print(f"  Experiment    : {MLFLOW_EXPERIMENT_NAME}")
print(f"\n  Notebook 02 champion R2 : {best_info['best_test_r2']:.4f}")
print(f"  Pipeline R2             : {captured.get('test_r2'):.4f}")
print(f"  Delta                   : {captured.get('test_r2',0) - best_info['best_test_r2']:+.4f}")
print("\nUse the presigned links below (ignore any generic mlflow link above):")
print_mlflow_presigned_links(pipeline_exp_id, pipeline_run_id)

**Reconciliation matters.** The pipeline R² should land within roughly ±0.01 of
Notebook 02's champion. A larger gap means the two code paths have diverged —
which is exactly the class of bug the shared `features.py` exists to prevent, so
a discrepancy here is a real signal, not noise to be waved through.

## 5. Approve and Deploy a Serverless Endpoint

The gate passed, so the model is in the registry as `PendingManualApproval`.
That manual step is deliberate: it is the human checkpoint where the model card
and the governance findings are reviewed before anything reaches users.

We deploy **serverless** — cost is near zero when idle, which suits an
intermittent host-facing pricing tool far better than a provisioned instance.

In [ ]:
sm = boto3.client("sagemaker")

pkgs = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1
)["ModelPackageSummaryList"]

if not pkgs:
    raise RuntimeError("No model packages found — did the Register step run? "
                       "If the gate failed, registration was correctly skipped.")

pkg_arn = pkgs[0]["ModelPackageArn"]
print(f"Latest model package : {pkg_arn}")
print(f"Current status       : {pkgs[0]['ModelApprovalStatus']}")
print(f"Created              : {pkgs[0]['CreationTime']}")

In [ ]:
# The human approval gate. In a real deployment this happens after model-card
# review and sign-off, not automatically as part of the notebook run.
sm.update_model_package(ModelPackageArn=pkg_arn, ModelApprovalStatus="Approved")
print(f"Approved: {pkg_arn}")
print("\nApproval checklist that should precede this in production:")
print("  [ ] Model card reviewed, including the known-limitations block")
print("  [ ] Spatial bias audit reviewed (§7 below)")
print("  [ ] Segment fairness reviewed (§7 below)")
print("  [ ] Serving guardrails confirmed present in inference.py")
print("  [ ] Rollback plan and monitoring thresholds agreed")

In [ ]:
from sagemaker import ModelPackage
from sagemaker.serverless import ServerlessInferenceConfig

deployable = ModelPackage(role=role, model_package_arn=pkg_arn,
                          sagemaker_session=sagemaker.Session())

serverless_cfg = ServerlessInferenceConfig(memory_size_in_mb=4096,
                                           max_concurrency=5)

print(f"Deploying serverless endpoint: {ENDPOINT_NAME}")
print("Takes roughly 3-5 minutes...")
predictor = deployable.deploy(serverless_inference_config=serverless_cfg,
                              endpoint_name=ENDPOINT_NAME)
print(f"\nEndpoint ready: {ENDPOINT_NAME}")
print("Cost: ~$0 when idle; billed per invocation.")

## 6. Test the Live Endpoint

Four probes chosen to exercise the guardrails, not just the happy path: a
mid-market listing the model handles well, a cold-start listing, a shared room
that should be refused, and a luxury listing at the edge of the calibrated range.

In [ ]:
rt = boto3.client("sagemaker-runtime")


def base_listing(**over):
    """A raw listing payload — the same schema the endpoint receives in prod."""
    d = {
        "city": "Paris", "neighbourhood": "Buttes-Montmartre",
        "latitude": 48.8867, "longitude": 2.3334,
        "property_type": "Entire apartment", "room_type": "Entire place",
        "accommodates": 4, "bedrooms": 2,
        "amenities": '["Wifi", "Kitchen", "Heating", "Washer", "TV", "Elevator"]',
        "minimum_nights": 2, "maximum_nights": 1125,
        "host_since": "2015-06-01", "host_location": "Paris, France",
        "host_response_time": "within an hour", "host_response_rate": 100.0,
        "host_acceptance_rate": 95.0, "host_is_superhost": "t",
        "host_total_listings_count": 2, "host_has_profile_pic": "t",
        "host_identity_verified": "t", "instant_bookable": "f",
        "review_scores_rating": 96.0, "review_scores_accuracy": 10.0,
        "review_scores_cleanliness": 9.0, "review_scores_checkin": 10.0,
        "review_scores_communication": 10.0, "review_scores_location": 10.0,
        "review_scores_value": 9.0,
    }
    d.update(over)
    return d


def call(payload, label):
    r = rt.invoke_endpoint(EndpointName=ENDPOINT_NAME,
                           ContentType="application/json",
                           Body=json.dumps(payload))
    res = json.loads(r["Body"].read())[0]
    print(f"\n{label}")
    print("-" * 66)
    if res["served"]:
        print(f"  Recommended : ${res['recommended_price_usd']}/night")
        print(f"  Range       : ${res['price_range_usd'][0]} - "
              f"${res['price_range_usd'][1]}  ({res['interval']})")
        print(f"  Confidence  : {res['confidence']}")
    else:
        print(f"  NOT SERVED  — confidence: {res['confidence']}")
    for n in res["notes"]:
        print(f"  note        : {n}")
    return res


r1 = call(base_listing(), "A. Mid-market Paris 2-bed (model's strongest case)")

r2 = call(base_listing(
    review_scores_rating=None, review_scores_accuracy=None,
    review_scores_cleanliness=None, review_scores_checkin=None,
    review_scores_communication=None, review_scores_location=None,
    review_scores_value=None, host_since="2021-01-15",
    host_total_listings_count=1, host_is_superhost="f"),
    "B. Brand-new Paris listing, no reviews (cold start)")

r3 = call(base_listing(
    city="Bangkok", neighbourhood="Bang Rak",
    latitude=13.7280, longitude=100.5240,
    room_type="Shared room", property_type="Shared room in hostel",
    accommodates=1, bedrooms=1),
    "C. Bangkok shared room (guardrail: should be REFUSED)")

r4 = call(base_listing(
    city="New York", neighbourhood="Chelsea",
    latitude=40.7465, longitude=-74.0014,
    accommodates=8, bedrooms=4, minimum_nights=30,
    amenities='["Wifi","Kitchen","Air conditioning","Elevator","Gym","Pool",'
              '"Dishwasher","Washer","Dryer","TV","Hot tub","Bathtub"]',
    host_total_listings_count=25),
    "D. Luxury NYC 4-bed, 30-night minimum (edge of calibrated range)")

The endpoint behaves as designed:

- **A** returns a tight range with `high` confidence — the model's home turf.
- **B** returns a wider range, `medium`/`low` confidence and an explicit
  cold-start note, reflecting the measured 31% MdAPE for never-reviewed listings.
- **C** is **refused outright**. The endpoint declines rather than emitting a
  number it knows carries ~44% typical error. This is the error analysis
  functioning as a control, not as a footnote.
- **D** returns a value with an edge-of-range warning, since decile-10 listings
  are systematically under-priced by the model.

## 7. AI Governance Checks

Pipeline stage 9. These are not a formality appended after deployment — two of
the three findings below would justify blocking a real launch.

### 7.1 Spatial bias audit

Notebook 02 established that geography accounts for roughly **60% of the model's
predictive power**. A model that is largely a location-price index will
faithfully reproduce whatever spatial price structure exists in its training
data — including structure that reflects historical segregation, tourism
gentrification, or discriminatory pricing.

The question is not "does the model use location?" — it must, location genuinely
determines accommodation prices. The question is whether the model **amplifies**
existing spatial inequality by predicting a wider gap than actually exists.

In [ ]:
# Re-load the held-out test set produced by the pipeline for the audit
proc_uri = f"{PIPELINE_ROOT}/processed"


def read_pipeline_csv(name):
    key = f"{PREFIX}/pipeline/processed/{name}"
    obj = s3_client.get_object(Bucket=BUCKET, Key=key)
    return pd.read_csv(io.BytesIO(obj["Body"].read()))


X_test_p = read_pipeline_csv("test_features.csv")
y_test_p = read_pipeline_csv("test_labels.csv").squeeze("columns")
meta_p   = read_pipeline_csv("test_meta.csv")

# Pull the trained Pipeline out of the model artefact for local auditing
import tarfile, pickle
local_tar = "model.tar.gz"
b, k = model_artifact_uri.replace("s3://", "").split("/", 1)
s3_client.download_file(b, k, local_tar)
with tarfile.open(local_tar) as t:
    t.extractall("model_extract")
with open("model_extract/model.pkl", "rb") as f:
    prod_model = pickle.load(f)

pred_p = prod_model.predict(X_test_p)
aud = meta_p.copy()
aud["actual"] = y_test_p.values
aud["pred"]   = pred_p
aud["resid"]  = aud.actual - aud.pred
aud["ape"]    = np.abs(np.exp(aud.pred) - np.exp(aud.actual)) / np.exp(aud.actual)

from sklearn.metrics import r2_score
print(f"Production model reproduced locally — test R2 = "
      f"{r2_score(aud.actual, aud.pred):.4f}")

In [ ]:
# --- amplification test: predicted vs actual spread across neighbourhoods ---
nb = (aud.groupby(["city", "neighbourhood"])
        .agg(n=("actual", "size"),
             actual_med=("actual", "median"),
             pred_med=("pred", "median"),
             bias=("resid", "mean"),
             mdape=("ape", "median"))
        .query("n >= 100"))

actual_spread = nb.actual_med.max() - nb.actual_med.min()
pred_spread   = nb.pred_med.max() - nb.pred_med.min()
amplification = pred_spread / actual_spread

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(nb.actual_med, nb.pred_med, s=nb.n / 6, alpha=.55, color="#5B9BD5")
lims = [nb.actual_med.min() - .1, nb.actual_med.max() + .1]
axes[0].plot(lims, lims, "r--", label="no amplification")
axes[0].set_xlabel("actual median log price"); axes[0].set_ylabel("predicted median")
axes[0].set_title("Neighbourhood-level calibration", fontweight="bold")
axes[0].legend()

axes[1].scatter(nb.actual_med, nb.bias, s=nb.n / 6, alpha=.55, color="#E67E22")
axes[1].axhline(0, c="red", ls="--")
axes[1].set_xlabel("actual median log price"); axes[1].set_ylabel("mean residual")
axes[1].set_title("Bias vs neighbourhood price level", fontweight="bold")
plt.tight_layout(); plt.show()

corr_bias = np.corrcoef(nb.actual_med, nb.bias)[0, 1]
print("SPATIAL BIAS AUDIT")
print("=" * 66)
print(f"  Neighbourhoods audited (n>=100) : {len(nb)}")
print(f"  Actual log-price spread         : {actual_spread:.3f}")
print(f"  Predicted log-price spread      : {pred_spread:.3f}")
print(f"  AMPLIFICATION RATIO             : {amplification:.3f}")
print(f"    (>1 = model exaggerates spatial inequality)")
print(f"  corr(neighbourhood price, bias) : {corr_bias:+.3f}")
print(f"    (negative = cheap areas over-priced, expensive under-priced)")
print("=" * 66)

**Reading the audit.** The amplification ratio below 1.0 shows the model
**compresses** rather than exaggerates spatial price differences — a direct
consequence of the regression-to-the-mean effect from Notebook 02 §9.1, and the
reassuring direction to find it in. The model is not widening the gap between
expensive and cheap neighbourhoods.

But the negative `corr(neighbourhood price, bias)` carries a real equity concern
that runs the *other* way:

> Listings in **low-priced neighbourhoods are systematically told to charge
> more** than the local market bears, and listings in high-priced neighbourhoods
> are told to charge less.

For a host in a cheap neighbourhood, following the recommendation risks pricing
out of their local market and losing bookings — a concrete financial harm that
falls disproportionately on hosts in lower-income areas. Statistical
compression is the correct behaviour for a squared-error model; it is not the
correct behaviour for a product. **This is the strongest argument for the
`price_range_usd` band rather than a bare point estimate**, and for the
edge-of-range warning already implemented in `inference.py`.

### 7.2 Segment fairness — equality of service quality

In [ ]:
def fairness_table(df, col, label):
    t = df.groupby(col).agg(n=("ape", "size"), mdape=("ape", "median"),
                            bias=("resid", "mean"))
    t["disparity_vs_best"] = (t.mdape / t.mdape.min()).round(2)
    t.index.name = label
    return t.sort_values("mdape")


print("FAIRNESS: DISPARITY IN SERVICE QUALITY ACROSS GROUPS")
print("=" * 74)
for col, lbl in [("city", "City"), ("room_type", "Room type"),
                 ("has_reviews", "Has reviews"),
                 ("is_professional_host", "Professional host")]:
    t = fairness_table(aud, col, lbl)
    print(f"\n{lbl}")
    print(t.round(3).to_string())
    worst = t.disparity_vs_best.max()
    flag = "FAIL" if worst > 1.5 else "WATCH" if worst > 1.25 else "OK"
    print(f"  -> max disparity {worst:.2f}x  [{flag}]")

print("\n" + "=" * 74)
print("Threshold convention: >1.5x disparity in median error between the")
print("best- and worst-served group is treated as a launch blocker for that")
print("segment; 1.25-1.5x requires a documented mitigation.")

**Findings and required actions.**

| Group | Disparity | Verdict | Action taken |
|---|---|---|---|
| Room type (Shared vs Entire) | ~1.8× | **FAIL** | Segment refused at the endpoint |
| City (Hong Kong vs Paris) | ~1.6× | **FAIL** | Per-city confidence tiers; wider bands |
| Review status | ~1.3× | WATCH | Cold-start penalty on interval width |
| Host type | ~1.2× | OK | Documented, monitored |

The important point is that these disparities are in **precision, not
direction** — the bias term is near zero for every group. The model is not
systematically under-pricing any protected or vulnerable segment; it is simply
*less certain* for some of them. That is a materially different — and more
tractable — problem than directional discrimination, and honest uncertainty
communication is a genuine fix for it rather than a fig leaf.

**Protected attributes.** This dataset contains no direct protected attributes
(race, gender, age, nationality). It does contain `neighbourhood`, which is a
well-known proxy for race and income in several of these cities. That is why
§7.1 is a launch-blocking check rather than an appendix.

### 7.3 Interpretability and the drift-monitoring baseline

In [ ]:
from sklearn.inspection import permutation_importance

sub = np.random.RandomState(42).choice(len(X_test_p), min(15000, len(X_test_p)),
                                       replace=False)
pi = permutation_importance(prod_model, X_test_p.iloc[sub], y_test_p.iloc[sub],
                            n_repeats=3, random_state=42, scoring="r2", n_jobs=2)
imp = pd.Series(pi.importances_mean, index=X_test_p.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
imp.head(18)[::-1].plot(kind="barh", ax=ax, color="#5B9BD5")
ax.set_xlabel("drop in test R2 when shuffled")
ax.set_title("Production model — permutation importance (governance record)",
             fontweight="bold")
plt.tight_layout(); plt.show()

geo_share = (imp.get("city_neigh", 0) + imp.get("city", 0) +
             imp.get("dist_center_km", 0) + imp.get("lat_offset", 0) +
             imp.get("lon_offset", 0)) / imp[imp > 0].sum()
print(f"Geography's share of positive importance: {geo_share:.1%}")
print("\nTop 12:"); print(imp.head(12).round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Drift baseline — pipeline stage 8 (Monitoring & Logging).
# Snapshot the reference distribution of the features that matter most, so a
# scheduled job can compare live traffic against it. Without a baseline
# captured at deployment time, drift detection is retrospective guesswork.
# ---------------------------------------------------------------------------
monitor_features = imp.head(12).index.tolist()

baseline = {
    "captured_utc": pd.Timestamp.utcnow().isoformat(),
    "endpoint": ENDPOINT_NAME,
    "model_package_arn": pkg_arn,
    "training_job": training_job_name,
    "n_reference_rows": int(len(X_test_p)),
    "performance": {
        "test_r2": float(captured.get("test_r2", np.nan)),
        "test_mdape": float(captured.get("test_mdape", np.nan)),
    },
    "alert_thresholds": {
        "r2_floor": QUALITY_GATE_R2,
        "mdape_ceiling": 0.32,
        "psi_warn": 0.10,
        "psi_alert": 0.25,
        "note": "PSI computed per feature against the quantiles below.",
    },
    "feature_quantiles": {
        f: {q: float(X_test_p[f].quantile(v))
            for q, v in [("p05", .05), ("p25", .25), ("p50", .50),
                         ("p75", .75), ("p95", .95)]}
        for f in monitor_features if pd.api.types.is_numeric_dtype(X_test_p[f])
    },
    "categorical_distribution": {
        f: X_test_p[f].value_counts(normalize=True).head(15).round(4).to_dict()
        for f in ["city", "room_type", "property_group"] if f in X_test_p.columns
    },
    "prediction_distribution": {
        "log_p05": float(np.quantile(pred_p, .05)),
        "log_p50": float(np.quantile(pred_p, .50)),
        "log_p95": float(np.quantile(pred_p, .95)),
        "usd_median": float(np.exp(np.quantile(pred_p, .50))),
    },
    "known_limitations": best_info["known_limitations"],
    "spatial_amplification_ratio": float(amplification),
    "geography_importance_share": float(geo_share),
    "drift_triggers_to_watch": [
        "NYC minimum_nights=30 share changes -> Multiple Dwelling Law amended",
        "FX table staleness -> rates are fixed at Feb/Mar 2021 values",
        "host_acceptance_rate missingness rate -> platform policy change",
        "post-COVID demand recovery -> absolute price levels shift upward",
    ],
}

Path("monitoring_baseline.json").write_text(json.dumps(baseline, indent=2))
s3_client.put_object(Bucket=BUCKET,
                     Key=f"{PREFIX}/monitoring/monitoring_baseline.json",
                     Body=json.dumps(baseline, indent=2))

with mlflow.start_run(run_name=f"{TEAM_ID}_{STUDENT_ID}_governance_baseline") as run:
    mlflow.set_tags({"course": COURSE, "team_id": TEAM_ID, "student_id": STUDENT_ID,
                     "run_role": "governance_and_monitoring",
                     "tracking_backend": "sagemaker_mlflow_app"})
    mlflow.log_metrics({
        "spatial_amplification_ratio": float(amplification),
        "neighbourhood_bias_correlation": float(corr_bias),
        "geography_importance_share": float(geo_share),
        "max_city_disparity": float(
            fairness_table(aud, "city", "City").disparity_vs_best.max()),
        "max_roomtype_disparity": float(
            fairness_table(aud, "room_type", "Room").disparity_vs_best.max()),
    })
    mlflow.log_artifact("monitoring_baseline.json", artifact_path="governance")

print("Monitoring baseline captured and logged.")
print(f"  s3://{BUCKET}/{PREFIX}/monitoring/monitoring_baseline.json")
print(f"\n  Alert if test R2 < {QUALITY_GATE_R2} or MdAPE > 0.32 on fresh data.")
print(f"  Alert if PSI > 0.25 on any of: {monitor_features[:5]} ...")

## 8. Retraining and Rollback

The pipeline is **parameterised and idempotent**, so a retrain is a single call.
Because the `ConditionStep` gates registration on R², a degraded retrain cannot
reach the registry — the previous approved package remains the deployable
artefact, which is the rollback path.

In [ ]:
retrain_snippet = "\n".join([
    "# Retrain on refreshed data (e.g. a new monthly snapshot in S3).",
    "# Nothing needs editing: the pipeline reads from RAW_DATA_URI and the",
    "# gate protects the registry. Run from a schedule, a Lambda, or CI.",
    "",
    "from sagemaker.workflow.pipeline import Pipeline",
    f'pipeline = Pipeline(name="{PIPELINE_NAME}")',
    "execution = pipeline.start(parameters={",
    '    "MaxIter": 600, "LearningRate": 0.06, "MaxLeafNodes": 63,',
    '    "MinSamplesLeaf": 40, "L2Regularization": 1.0,',
    f'    "QualityGateR2": {QUALITY_GATE_R2},',
    "})",
    "execution.wait()",
    "",
    "# Rollback: re-point the endpoint at the previously approved package.",
    f'# sm.update_endpoint(EndpointName="{ENDPOINT_NAME}",',
    "#                    EndpointConfigName=<previous_config>)",
])
print(retrain_snippet)

print("\nRETRAINING TRIGGERS")
print("=" * 66)
for t in ["Scheduled: monthly, on the new listings snapshot",
          "Performance: rolling MdAPE on labelled bookings > 0.32",
          "Drift: PSI > 0.25 on any monitored feature",
          "Event: FX table revision (rates are fixed at Feb/Mar 2021)",
          "Event: short-term-rental regulation change in any covered city"]:
    print(f"  - {t}")

## 9. Clean Up

Serverless endpoints cost nothing when idle, but leaving them running still
accrues per-invocation charges and occupies account quota. Delete when done.

In [ ]:
# Uncomment to tear down.
# boto3.client("sagemaker").delete_endpoint(EndpointName=ENDPOINT_NAME)
# print(f"Endpoint deleted: {ENDPOINT_NAME}")
print(f"Endpoint {ENDPOINT_NAME} left running. Uncomment above to delete.")

In [ ]:
print("=" * 70)
print("NOTEBOOK 03 COMPLETE")
print("=" * 70)
print(f"Pipeline        : {PIPELINE_NAME}")
print(f"Execution       : {execution.arn.rsplit('/', 1)[-1]}")
print(f"Training job    : {training_job_name}")
print(f"Test R2         : {captured.get('test_r2')}  (gate {QUALITY_GATE_R2})")
print(f"Test MdAPE      : {captured.get('test_mdape')}")
print(f"Model registry  : {MODEL_PACKAGE_GROUP}")
print(f"Endpoint        : {ENDPOINT_NAME} (serverless)")
print(f"MLflow          : {MLFLOW_EXPERIMENT_NAME}")
print(f"Monitoring      : s3://{BUCKET}/{PREFIX}/monitoring/")
print()
print("Governance summary")
print(f"  spatial amplification : {amplification:.3f} (<1 = compresses, good)")
print(f"  geography importance  : {geo_share:.1%}")
print(f"  segments refused      : Shared room (MdAPE ~44%)")
print(f"  low-confidence tiers  : Hong Kong, Istanbul, Rio; all cold-start")
print()
print("Next: the Final Technical Report.")

---

## Checklist

- [ ] `features.py` shared by preprocessing, training **and** inference
- [ ] Training/serving skew asserted absent before any SageMaker job was launched
- [ ] Pipeline source versioned in S3 and re-downloaded before use
- [ ] `SmoothedTargetEncoder` defined in a module so the artefact unpickles at the endpoint
- [ ] SageMaker Pipeline upserted and executed end to end
- [ ] `ConditionStep` gates registration on **test R² ≥ 0.70** (regression, not AUC)
- [ ] No MLflow dependency or credential inside any container
- [ ] Production run logged to the team MLflow App and reconciled against Notebook 02
- [ ] Model registered `PendingManualApproval`; approval treated as a review gate
- [ ] Serverless endpoint deployed and probed on four scenarios including a refusal
- [ ] Spatial bias audit run — amplification ratio and bias correlation recorded
- [ ] Segment fairness disparities measured against an explicit threshold
- [ ] Drift-monitoring baseline captured at deployment time and stored in S3
- [ ] Retraining triggers and rollback path documented

### The production artefact in one line

```text
raw listing JSON -> [ clean -> engineer -> one-hot + OOF target encode -> HGB ]
                 -> log price -> exp() -> median USD + 80% band + confidence tier
                 -> guardrails: refuse shared rooms, flag cold start, warn at extremes
```

One pickled `sklearn.Pipeline`. Same object in the training job and behind the
endpoint. No parallel serving-side preprocessing code, and therefore no
opportunity for it to drift.